# 17: How Does the Crystal Respond to Perturbation?

The notebook separates causal effect from coupling-dependent pathwise divergence. Cell-keyed CRN is an experimental coupling, not a replacement for Digital Crystal v1.


In [1]:
from pathlib import Path
import json
import math
import random
import sys
from collections import Counter, defaultdict

import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
BOOK_FIG_DIR = ROOT / "static" / "images" / "books" / "digital-life"
FIG_DIR = ROOT / "notebooks" / "generated-figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(ROOT / "notebooks" / "_shared"))

from digital_crystal import (
    CrystalParams, CrystalState, neighbors, hex_distance, hex_capacity,
    axial_to_xy, logistic, local_exposure_angle, initial_state, clone_state,
    morphology_hash, frontier, attachment_probability, advance_one_step,
    run_crystal, cell_keyed_uniform,
)

plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 180

NOTEBOOK_PROFILE = "quick"
RUN_CANONICAL = False

def load_json(path):
    return json.loads((ROOT / path).read_text(encoding="utf-8"))

def audit(expected, observed, source):
    return {
        "source": source,
        "expected": expected,
        "observed": observed,
        "matches": {k: observed.get(k) == v for k, v in expected.items()},
    }

def plot_cells(cells, title, ax=None, color="tab:blue"):
    ax = ax or plt.gca()
    pts = np.array([axial_to_xy(c) for c in cells]) if cells else np.empty((0, 2))
    if len(pts):
        ax.scatter(pts[:, 0], pts[:, 1], marker="h", s=42, c=color, alpha=0.9)
    ax.set_title(title)
    ax.set_aspect("equal")
    ax.axis("off")
    return ax


In [2]:
EXPERIMENT = {"seed": 17, "max_radius": 24, "steps": 24, "pulse_step": 5, "pulse_gain": 0.8}
EXPERIMENT


{'seed': 17, 'max_radius': 24, 'steps': 24, 'pulse_step': 5, 'pulse_gain': 0.8}

## Sequential RNG coupling vs cell-keyed CRN


In [3]:
def advance_one_step_keyed(state, input_value, max_radius, seed, params=CrystalParams()):
    occupied = set(state.occupied); birth_time = dict(state.birth_time); additions = []
    for cell in sorted(frontier(occupied, max_radius)):
        p = attachment_probability(cell, occupied, input_value, params)
        if cell_keyed_uniform(seed, state.step + 1, cell) < p:
            additions.append(cell)
    step = state.step + 1
    for cell in additions:
        occupied.add(cell); birth_time[cell] = step
    return CrystalState(occupied, birth_time, step, state.rng_state, state.attachments_by_step + [len(additions)], state.population_by_step + [len(occupied)])

def fork_compare(kind):
    a = initial_state(100); b = initial_state(100)
    for t in range(EXPERIMENT["steps"]):
        va = 0.4 + (EXPERIMENT["pulse_gain"] if t == EXPERIMENT["pulse_step"] else 0)
        vb = 0.4
        if kind == "sequential":
            a, _ = advance_one_step(a, va, EXPERIMENT["max_radius"])
            b, _ = advance_one_step(b, vb, EXPERIMENT["max_radius"])
        else:
            a = advance_one_step_keyed(a, va, EXPERIMENT["max_radius"], EXPERIMENT["seed"])
            b = advance_one_step_keyed(b, vb, EXPERIMENT["max_radius"], EXPERIMENT["seed"])
    return len(a.occupied ^ b.occupied), len(a.occupied), len(b.occupied)
{"sequential": fork_compare("sequential"), "cell_keyed_crn": fork_compare("crn")}


{'sequential': (205, 678, 707), 'cell_keyed_crn': (21, 841, 824)}

## Canonical artifact audit


In [4]:
preflight = load_json("research/digital-life/ch17-perturbation-dynamics-v6/stage-00-confirmatory-preflight.json")
matched = load_json("research/digital-life/ch17-perturbation-dynamics-v6/stage-01-confirmatory-matched-arrangement.json")
verdict = load_json("research/digital-life/ch17-perturbation-dynamics-v6/stage-02-confirmatory-verdict.json")
coupling = load_json("research/digital-life/ch17-perturbation-dynamics-v5/stage-01-randomness-coupling-audit.json")
floor = load_json("research/digital-life/ch17-perturbation-dynamics-v5/stage-02-crn-superposition-with-floor.json")
observed = {
    "preflight_p": round(preflight["omnibus_marginal_feature_test"]["p_value"], 3),
    "all_equivalence_checks_passed": preflight["all_equivalence_checks_passed"],
    "pulse_count_A": matched["codeword_validation"]["pulse_count_A"],
    "pulse_count_B": matched["codeword_validation"]["pulse_count_B"],
    "primary_symdiff_mean": round(matched["results"]["8"]["symdiff"]["mean"], 3),
    "primary_p": round(verdict["primary_p_value"], 4),
    "secondary_all24_p": round(matched["results"]["8"]["secondary_all24"]["p_value"], 4),
    "matched_arrangement_status": verdict["matched_arrangement_status"],
}
EXPECTED = {
    "preflight_p": 0.922,
    "all_equivalence_checks_passed": True,
    "pulse_count_A": 4,
    "pulse_count_B": 4,
    "primary_symdiff_mean": 0.053,
    "primary_p": 0.7366,
    "secondary_all24_p": 0.9320,
    "matched_arrangement_status": "FAILED",
}
audit(EXPECTED, observed, "canonical report artifact")


{'source': 'canonical report artifact',
 'expected': {'preflight_p': 0.922,
  'all_equivalence_checks_passed': True,
  'pulse_count_A': 4,
  'pulse_count_B': 4,
  'primary_symdiff_mean': 0.053,
  'primary_p': 0.7366,
  'secondary_all24_p': 0.932,
  'matched_arrangement_status': 'FAILED'},
 'observed': {'preflight_p': 0.922,
  'all_equivalence_checks_passed': True,
  'pulse_count_A': 4,
  'pulse_count_B': 4,
  'primary_symdiff_mean': 0.053,
  'primary_p': 0.7366,
  'secondary_all24_p': 0.932,
  'matched_arrangement_status': 'FAILED'},
 'matches': {'preflight_p': True,
  'all_equivalence_checks_passed': True,
  'pulse_count_A': True,
  'pulse_count_B': True,
  'primary_symdiff_mean': True,
  'primary_p': True,
  'secondary_all24_p': True,
  'matched_arrangement_status': True}}

In [5]:
seq20 = coupling["summary"]["sequential_causal"]["20"]["symdiff"]["mean"]
crn20 = coupling["summary"]["crn_causal"]["20"]["symdiff"]["mean"]
ind20 = coupling["summary"]["independent_reseed_reference"]["20"]["symdiff"]["mean"]
superposition_residual = floor["summary"]["8"]["clustered"]["residual_norm"]["value"]
noise_floor = floor["summary"]["8"]["zero_response_population_mean_noise_floor"]["mean"]
{"sequential_fraction_of_independent": seq20 / ind20, "crn_fraction_of_independent": crn20 / ind20, "superposition_residual": superposition_residual, "measurement_floor": noise_floor}


{'sequential_fraction_of_independent': 0.8673057742870616,
 'crn_fraction_of_independent': 0.10732151337263716,
 'superposition_residual': 0.007462343876518504,
 'measurement_floor': 0.04518632890073799}

## Bounded conclusion

Different histories produce different particular futures under CRN, but the tested population-level morphology signature failed. Earlier nonlinear interpretation also failed once compared with the measurement floor. Do not say the crystal forgot; the past can contribute causally without remaining legible in measured present morphology.

## Provenance

Book chapter: `content/books/digital-life/17-how-does-the-crystal-respond-to-perturbation/index.md`

Research scripts: `ch17_digital_crystal_perturbation_dynamics_v1..v6.py`; information-survival scripts inspected as related but not primary current evidence.

Canonical artifacts: `research/digital-life/ch17-perturbation-dynamics-v5/`, `research/digital-life/ch17-perturbation-dynamics-v6/`

Execution mode: quick-recomputed + canonical-artifact-audit
